In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder,
    MinMaxScaler, StandardScaler
)
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df = pd.read_csv('/content/MyDrive/ML/Virat_Kohli (1).csv')

print("First five rows:")
print(df.head())

print("\nDataset information:")
df.info()

print("\nStatistical description:")
print(df.describe(include='all'))

print("\nMissing values before preprocessing:")
print(df.isnull().sum())


df['Runs'] = pd.to_numeric(
    df['Runs'].astype(str).str.strip().str.replace('*', '', regex=False),
    errors='coerce'
)

df['Date'] = pd.to_datetime(
    df['Date'], dayfirst=True, errors='coerce'
)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

df['Runs'] = df['Runs'].fillna(df['Runs'].median())
df['Ground'] = df['Ground'].fillna(df['Ground'].mode()[0])

df['Year'] = df['Year'].fillna(df['Year'].median())
df['Month'] = df['Month'].fillna(df['Month'].mode()[0])

print("\nMissing values after handling:")
print(df.isnull().sum())

label_encoder = LabelEncoder()
df['Result_Encoded'] = label_encoder.fit_transform(
    df['Result']
)

print("\nLabel Encoding:")
print(df[['Result', 'Result_Encoded']].drop_duplicates())


X = df[
    ['Runs', 'Position', 'Innings', 'Year', 'Month',
     'Against', 'Venue', 'Ground']
]

y = df['Result']

numeric_features = [
    'Runs', 'Position', 'Innings', 'Year', 'Month'
]

categorical_features = ['Against', 'Venue', 'Ground']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nOriginal feature shape:", X.shape)
print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)

minmax = MinMaxScaler()
normalized = minmax.fit_transform(df[numeric_features])

normalized_df = pd.DataFrame(
    normalized,
    columns=[col + '_Normalized' for col in numeric_features]
)

print("\nNormalized data:")
print(normalized_df.head())

standard_scaler = StandardScaler()
standardized = standard_scaler.fit_transform(
    df[numeric_features]
)

standardized_df = pd.DataFrame(
    standardized,
    columns=[col + '_Standardized' for col in numeric_features]
)

print("\nStandardized data:")
print(standardized_df.head())

plt.figure(figsize=(8, 4))
plt.plot(df['Runs'].values, label='Original Runs')
plt.plot(
    normalized_df['Runs_Normalized'].values,
    label='Normalized Runs'
)
plt.xlabel("Record Number")
plt.ylabel("Value")
plt.title("Original vs Normalized Runs")
plt.legend()
plt.tight_layout()
plt.show()

print("\nData preprocessing completed successfully.")
